# 🧥 MeshVTON — IDM-VTON Baseline (ÇALIŞAN 2D Try-On)

Bu notebook, gerçek IDM-VTON pipeline'ını doğru ön-işleme (densepose + parsing mask) ile
çalıştırır ve **temiz 2D try-on** üretir. Faz 0 doğrulandı ✅.

Sıra:
1. GPU + repolar + kütüphaneler (detectron2 dahil — kaynaktan derlenir, ~15 dk)
2. Ön-işleme model dosyalarını indir (densepose / humanparsing / openpose)
3. Gerçek IDM-VTON pipeline'ını yükle
4. Kişi + kıyafet ver → densepose + mask → temiz try-on

> Sonraki faz: ControlNet3D (3D mesh koşullandırma) bu çalışan pipeline'a eklenecek.

## 1️⃣ GPU

In [ ]:
import torch
print('CUDA:', torch.cuda.is_available(), '| torch', torch.__version__)
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

## 2️⃣ Repoları klonla

In [ ]:
import os
# Bizim repo (src.idm_vton + ControlNet3D burada)
if not os.path.exists('/content/MeshVTON'):
    !git clone https://github.com/SerhanTelatar/MeshVTON.git /content/MeshVTON
else:
    !cd /content/MeshVTON && git pull

# IDM-VTON resmi repo (preprocess: densepose/parsing/openpose + apply_net + configs)
if not os.path.exists('/content/IDM-VTON-official'):
    !git clone https://github.com/yisol/IDM-VTON.git /content/IDM-VTON-official
print('✅ repolar hazır')

## 3️⃣ Kütüphaneler (+ detectron2 kaynaktan, ~15 dk)

In [ ]:
import os
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["FORCE_CUDA"] = "1"

# IDM-VTON ile uyumlu sabit sürümler
!pip install -q diffusers==0.25.0 transformers==4.36.2 accelerate==0.25.0 huggingface_hub==0.20.3 peft==0.7.1
!pip install -q omegaconf opencv-python-headless einops onnxruntime-gpu av ninja

# detectron2 (densepose için) — torch 2.11'e karşı kaynaktan derlenir
!pip install 'git+https://github.com/facebookresearch/detectron2.git'

import detectron2
from detectron2 import _C  # compiled ops çalışıyor mu
print('✅ detectron2', detectron2.__version__)

## 4️⃣ Ön-işleme model dosyalarını indir

In [ ]:
import os, shutil
from huggingface_hub import hf_hub_download

base = '/content/IDM-VTON-official/ckpt'

# densepose checkpoint (GitHub LFS çalışmıyor → resmi CDN'den)
dp = f'{base}/densepose/model_final_162be9.pkl'
os.makedirs(os.path.dirname(dp), exist_ok=True)
if not os.path.exists(dp) or os.path.getsize(dp) < 1_000_000:
    !wget -q https://dl.fbaipublicfiles.com/densepose/densepose_rcnn_R_50_FPN_s1x/165712039/model_final_162be9.pkl -O {dp}
print('densepose:', round(os.path.getsize(dp)/1e6,1), 'MB')

# humanparsing + openpose (HF'den)
for rf, dest in {
    'humanparsing/parsing_atr.onnx':      f'{base}/humanparsing/parsing_atr.onnx',
    'humanparsing/parsing_lip.onnx':      f'{base}/humanparsing/parsing_lip.onnx',
    'openpose/ckpts/body_pose_model.pth': f'{base}/openpose/ckpts/body_pose_model.pth',
}.items():
    if not os.path.exists(dest) or os.path.getsize(dest) < 1_000_000:
        f = hf_hub_download('yisol/IDM-VTON', rf)
        os.makedirs(os.path.dirname(dest), exist_ok=True); shutil.copy(f, dest)
    print(rf, '->', round(os.path.getsize(dest)/1e6,1), 'MB')
print('✅ ön-işleme modelleri hazır')

## 5️⃣ Gerçek IDM-VTON pipeline'ını yükle

In [ ]:
import sys, torch
for m in list(sys.modules):
    if m == 'src' or m.startswith('src.'):
        del sys.modules[m]
sys.path.insert(0, '/content/MeshVTON')

from src.idm_vton.tryon_pipeline import StableDiffusionXLInpaintPipeline as TryonPipeline
from src.idm_vton.unet_hacked_tryon import UNet2DConditionModel
from src.idm_vton.unet_hacked_garmnet import UNet2DConditionModel as UNet2DConditionModel_ref
from transformers import (CLIPImageProcessor, CLIPVisionModelWithProjection,
                          CLIPTextModel, CLIPTextModelWithProjection, AutoTokenizer)
from diffusers import DDPMScheduler, AutoencoderKL

base = 'yisol/IDM-VTON'; dt = torch.float16
unet          = UNet2DConditionModel.from_pretrained(base, subfolder="unet", torch_dtype=dt); unet.requires_grad_(False)
UNet_Encoder  = UNet2DConditionModel_ref.from_pretrained(base, subfolder="unet_encoder", torch_dtype=dt); UNet_Encoder.requires_grad_(False)
vae           = AutoencoderKL.from_pretrained(base, subfolder="vae", torch_dtype=dt)
image_encoder = CLIPVisionModelWithProjection.from_pretrained(base, subfolder="image_encoder", torch_dtype=dt)
text_encoder_one = CLIPTextModel.from_pretrained(base, subfolder="text_encoder", torch_dtype=dt)
text_encoder_two = CLIPTextModelWithProjection.from_pretrained(base, subfolder="text_encoder_2", torch_dtype=dt)
tokenizer_one = AutoTokenizer.from_pretrained(base, subfolder="tokenizer", use_fast=False)
tokenizer_two = AutoTokenizer.from_pretrained(base, subfolder="tokenizer_2", use_fast=False)
scheduler     = DDPMScheduler.from_pretrained(base, subfolder="scheduler")

pipe = TryonPipeline.from_pretrained(
    base, unet=unet, vae=vae, feature_extractor=CLIPImageProcessor(),
    text_encoder=text_encoder_one, text_encoder_2=text_encoder_two,
    tokenizer=tokenizer_one, tokenizer_2=tokenizer_two,
    scheduler=scheduler, image_encoder=image_encoder, torch_dtype=dt)
pipe.unet_encoder = UNet_Encoder
pipe = pipe.to('cuda'); pipe.unet_encoder.to('cuda')
print('✅ pipeline yüklendi')

## 6️⃣ Ön-işleme modüllerini hazırla (parsing / openpose)

In [ ]:
import sys
sys.path.insert(0, '/content/IDM-VTON-official')
sys.path.insert(0, '/content/IDM-VTON-official/gradio_demo')

from preprocess.humanparsing.run_parsing import Parsing
from preprocess.openpose.run_openpose import OpenPose
from utils_mask import get_mask_location
import apply_net
from detectron2.data.detection_utils import convert_PIL_to_numpy, _apply_exif_orientation

parsing_model  = Parsing(0)
openpose_model = OpenPose(0)
print('✅ parsing + openpose hazır')

## 7️⃣ Try-On çalıştır

Kişi + kıyafet görseli yükle. `category`: upper_body / lower_body / dresses.

In [ ]:
import torch
from PIL import Image
from torchvision import transforms
import matplotlib.pyplot as plt
from google.colab import files

dt = torch.float16; device = "cuda"
tfm = transforms.Compose([transforms.ToTensor(), transforms.Normalize([0.5],[0.5])])

print("👤 KİŞİ görseli:");   PERSON_PATH  = list(files.upload().keys())[0]
print("👕 KIYAFET görseli:"); GARMENT_PATH = list(files.upload().keys())[0]

garment_des = "a t-shirt"      # kıyafet açıklaması
category    = "upper_body"

human_img = Image.open(PERSON_PATH).convert("RGB").resize((768,1024))
garm_img  = Image.open(GARMENT_PATH).convert("RGB").resize((768,1024))

# maske
kp = openpose_model(human_img.resize((384,512)))
parse,_ = parsing_model(human_img.resize((384,512)))
mask,_ = get_mask_location('hd', category, parse, kp); mask = mask.resize((768,1024))

# densepose
arg_in = convert_PIL_to_numpy(_apply_exif_orientation(human_img.resize((384,512))), format="BGR")
args = apply_net.create_argument_parser().parse_args((
    'show','/content/IDM-VTON-official/configs/densepose_rcnn_R_50_FPN_s1x.yaml',
    '/content/IDM-VTON-official/ckpt/densepose/model_final_162be9.pkl',
    'dp_segm','-v','--opts','MODEL.DEVICE','cuda'))
pose_img = Image.fromarray(args.func(args, arg_in)[:,:,::-1]).resize((768,1024))

with torch.no_grad(), torch.cuda.amp.autocast():
    neg = "monochrome, lowres, bad anatomy, worst quality, low quality"
    pe,npe,ppe,nppe = pipe.encode_prompt("model is wearing "+garment_des,
        num_images_per_prompt=1, do_classifier_free_guidance=True, negative_prompt=neg)
    pe_c,_,_,_ = pipe.encode_prompt(["a photo of "+garment_des],
        num_images_per_prompt=1, do_classifier_free_guidance=False, negative_prompt=[neg])
    result = pipe(
        prompt_embeds=pe.to(device,dt), negative_prompt_embeds=npe.to(device,dt),
        pooled_prompt_embeds=ppe.to(device,dt), negative_pooled_prompt_embeds=nppe.to(device,dt),
        num_inference_steps=30, generator=torch.Generator(device).manual_seed(42), strength=1.0,
        pose_img=tfm(pose_img).unsqueeze(0).to(device,dt), text_embeds_cloth=pe_c.to(device,dt),
        cloth=tfm(garm_img).unsqueeze(0).to(device,dt), mask_image=mask, image=human_img,
        height=1024, width=768, ip_adapter_image=garm_img, guidance_scale=2.0)[0][0]

result.save('tryon_result.png')
fig,ax = plt.subplots(1,3,figsize=(15,7))
ax[0].imshow(human_img); ax[0].set_title('Kişi'); ax[0].axis('off')
ax[1].imshow(garm_img);  ax[1].set_title('Kıyafet'); ax[1].axis('off')
ax[2].imshow(result);    ax[2].set_title('Try-On'); ax[2].axis('off')
plt.tight_layout(); plt.show()